# 20.1 Integer variables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

By adding the keyword `integer` to the qualifying phrases of a `var` declaration, you
restrict the declared variables to `integer` values.

As an example, in analyzing the diet `model` in [Section 2.3](../02/2_3_using_the_AMPL_diet_model.ipynb), we arrived at the following
optimal solution:

```ampl
ampl: model diet.mod;
ampl: data diet2a.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

13 iterations, objective 118.0594032
ampl: display Buy;
Buy [*] :=
BEEF   5.36061
       CHK  2
FISH   2
       HAM 10
       MCH 10
       MTL 10
       SPG  9.30605
       TUR  2
;
```

If we want the foods to be purchased in integral amounts, we add `integer` to the
model's `var` declaration ([Figure 2-1](../02/2_2_an_AMPL_model_for_the_diet_problem.ipynb#fig-2-1)) as follows:

```ampl
var Buy {j in FOOD} integer >= f_min[j], <= f_max[j];
```

We can then try to re-`solve`:

```ampl
ampl: model dieti.mod; data diet2a.dat;
ampl: solve;
MINOS 5.5: ignoring integrality of 8 variables
MINOS 5.5: optimal solution found.

13 iterations, objective 118.0594032
```

As you can see, the MINOS solver does not handle integrality constraints. It has ignored
them and returned the same optimal value as before.

To get the `integer` optimum, we switch to a solver that does accommodate integrality:

```ampl
ampl: option solver cplex;
ampl: solve;
CPLEX 8.0.0: optimal integer solution; objective 119.3
11 MIP simplex iterations
1 branch-and-bound nodes
ampl: display Buy;
Buy [*] :=
BEEF   9
       CHK  2
FISH   2
       HAM  8
       MCH 10
       MTL 10
       SPG  7
       TUR  2
;
```

Comparing this solution to the previous one, we see a few features typical of `integer` programming.
The minimum cost has increased from $118.06 to $119.30; because integrality
is an additional constraint on the values of the variables, it can only make the `objective`
less favorable. The amounts of the different foods in the diet have also changed, but
in unpredictable ways. The two foods that had fractional amounts in the original optimum,
BEEF and SPG, have increased from 5.36061 to 9 and decreased from 9.30605 to 7,
respectively; also, HAM has dropped from the upper limit of 10 to 8. Clearly, you cannot
always deduce the `integer` optimum by rounding the non-`integer` optimum to the closest
`integer` values.